# SALT Whisper Submission Notebook

This notebook runs `huwenjie333/whisper-v3-ft-af51-0703` on the prepared WAXAL test audio and writes a Zindi-ready `ID,Target` submission CSV.

Compliance note: `docs/RULES_AND_DATA_USE.md` records an unresolved rule conflict around external data. Before treating this model as a final competition submission, confirm that the checkpoint license and training data are acceptable for the challenge.

Repository flow used here:

1. `scripts/prepare_dataset.py` creates `data/processed/hf_dataset` from the official Zindi IDs and matching WAXAL audio.
2. This notebook loads the prepared `test` split, forces the model language token from each ID's language, and writes raw predictions.
3. `waxal.submission.make_submission_file` aligns predictions exactly to `SampleSubmission.csv` and sanitizes empty or unsafe targets.
4. `scripts/validate_submission.py` checks row count, columns, ordering, duplicates, empty targets, and embedded newlines.

If `data/processed/hf_dataset` is missing, build the test cache first from the repo root:

```bash
uv run scripts/prepare_dataset.py \
  --raw-dir "$WAXAL_RAW_DIR" \
  --output-dir data/processed \
  --splits test \
  --languages lin sna lug \
  --skip-duration
```


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src" / "waxal").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()

sys.path.insert(0, str(REPO_ROOT / "src"))

RAW_DIR = Path(
    os.environ.get(
        "WAXAL_RAW_DIR",
        str(REPO_ROOT / "google-waxal-asr-challenge20260630-10570-elxebu"),
    )
).expanduser()
DATASET_DIR = REPO_ROOT / "data" / "processed"
HF_DATASET_DIR = DATASET_DIR / "hf_dataset"

print(f"Repo root: {REPO_ROOT}")
print(f"Raw Zindi dir: {RAW_DIR}")
print(f"Prepared dataset dir: {HF_DATASET_DIR}")


In [ ]:
MODEL_ID = "huwenjie333/whisper-v3-ft-af51-0703"
# Load the standard Whisper tokenizer/feature extractor. The fine-tuned repo's
# tokenizer config is not compatible with some recent Transformers builds.
PROCESSOR_ID = "openai/whisper-large-v3"
BATCH_SIZE = 4
MAX_NEW_TOKENS = 256
NUM_BEAMS = 1

# Leave as None for a full submission. Set a small integer for a quick smoke run.
MAX_SAMPLES = None

# WAXAL challenge target languages. These are the custom Whisper token IDs used by the SALT model.
SALT_LANGUAGE_TOKENS_WHISPER = {
    "lin": 50353,
    "lug": 50332,
    "sna": 50324,
}


In [ ]:
from collections import Counter

from datasets import load_from_disk

if not HF_DATASET_DIR.exists():
    raise FileNotFoundError(
        f"Missing prepared audio cache: {HF_DATASET_DIR}\n"
        "Build it from the repo root with:\n"
        "uv run scripts/prepare_dataset.py "
        f"--raw-dir {RAW_DIR} --output-dir data/processed "
        "--splits test --languages lin sna lug --skip-duration"
    )

dataset_dict = load_from_disk(HF_DATASET_DIR)
if "test" not in dataset_dict:
    raise KeyError(f"No test split found in {HF_DATASET_DIR}. Available splits: {list(dataset_dict)}")

test_ds = dataset_dict["test"]
if MAX_SAMPLES is not None:
    test_ds = test_ds.select(range(min(MAX_SAMPLES, len(test_ds))))

languages = sorted(set(test_ds["language"]))
unsupported = sorted(set(languages) - set(SALT_LANGUAGE_TOKENS_WHISPER))
if unsupported:
    raise ValueError(f"Missing SALT language token IDs for: {unsupported}")

print(test_ds)
print("Language counts:", dict(sorted(Counter(test_ds["language"]).items())))


In [ ]:
import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch_dtype = torch.float16 if device.type == "cuda" else torch.float32
print(f"Device: {device}; dtype: {torch_dtype}")

# Use the base Whisper processor because this fine-tuned repo has a tokenizer
# config that can fail on recent Transformers versions. The model weights still
# come from MODEL_ID, and SALT language IDs are forced manually below.
processor = WhisperProcessor.from_pretrained(PROCESSOR_ID)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    low_cpu_mem_usage=True,
)
model = model.to(device)
model.eval()

# Generation will receive explicit decoder prompts per language.
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

TRANSCRIBE_TOKEN = processor.tokenizer.convert_tokens_to_ids("<|transcribe|>")
NOTIMESTAMPS_TOKEN = processor.tokenizer.convert_tokens_to_ids("<|notimestamps|>")
if TRANSCRIBE_TOKEN is None or NOTIMESTAMPS_TOKEN is None:
    raise ValueError("Could not resolve Whisper transcribe/notimestamps tokens.")


def forced_decoder_ids_for_language(lang_code: str) -> list[tuple[int, int]]:
    return [
        (1, SALT_LANGUAGE_TOKENS_WHISPER[lang_code]),
        (2, TRANSCRIBE_TOKEN),
        (3, NOTIMESTAMPS_TOKEN),
    ]


In [ ]:
from tqdm.auto import tqdm

prediction_rows = []

for lang_code in languages:
    lang_ds = test_ds.filter(lambda row, lang=lang_code: row["language"] == lang)
    forced_decoder_ids = forced_decoder_ids_for_language(lang_code)
    print(f"Predicting {len(lang_ds)} {lang_code} rows")

    for start in tqdm(range(0, len(lang_ds), BATCH_SIZE), desc=lang_code):
        batch = lang_ds[start : start + BATCH_SIZE]
        audios = [audio["array"] for audio in batch["audio"]]

        inputs = processor(
            audios,
            sampling_rate=16_000,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
        )
        input_features = inputs.input_features.to(device=device, dtype=next(model.parameters()).dtype)
        attention_mask = inputs.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device=device)

        with torch.no_grad():
            predicted_ids = model.generate(
                input_features=input_features,
                attention_mask=attention_mask,
                forced_decoder_ids=forced_decoder_ids,
                num_beams=NUM_BEAMS,
                do_sample=False,
                max_new_tokens=MAX_NEW_TOKENS,
            )

        transcriptions = processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False,
        )
        prediction_rows.extend(
            {"ID": example_id, "Target": text.strip()}
            for example_id, text in zip(batch["ID"], transcriptions, strict=True)
        )

print(f"Predicted {len(prediction_rows)} rows")
prediction_rows[:3]


In [ ]:
from waxal.data import write_csv_rows
from waxal.utils import clean_name, utc_timestamp

run_name = f"{clean_name(MODEL_ID)}_salt_tokens"
if MAX_SAMPLES is not None:
    run_name = f"{run_name}_smoke_{MAX_SAMPLES}"

predictions_path = REPO_ROOT / "outputs" / "predictions" / f"{run_name}_test.csv"
write_csv_rows(predictions_path, prediction_rows, ["ID", "Target"])
print(f"Wrote predictions: {predictions_path}")


In [ ]:
from waxal.submission import make_submission_file

if MAX_SAMPLES is not None:
    submission_path = None
    print("MAX_SAMPLES is set, so this is a smoke run. Skipping final submission creation.")
else:
    submission_path = (
        REPO_ROOT
        / "outputs"
        / "submissions"
        / f"submission_{run_name}_{utc_timestamp()}.csv"
    )
    result = make_submission_file(
        predictions_path=predictions_path,
        raw_dir=RAW_DIR,
        output_path=submission_path,
        fill_missing="",
        sanitize=True,
        empty_fallback=".",
    )
    print(result)


In [ ]:
import subprocess

if submission_path is not None:
    completed = subprocess.run(
        [
            sys.executable,
            str(REPO_ROOT / "scripts" / "validate_submission.py"),
            "--submission",
            str(submission_path),
            "--raw-dir",
            str(RAW_DIR),
        ],
        cwd=REPO_ROOT,
        text=True,
        capture_output=True,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        print(completed.stderr)
        raise RuntimeError("Submission validation failed")
